In [ ]:
pip install thop torchinfo

In [ ]:
import torch
import time
import os
from thop import profile

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def compute_model_stats(model, input_size=(1, 3, 224, 224)):
    model = model.to(device)
    model.eval()

    dummy_input = torch.randn(input_size).to(device)

    # -----------------------
    # 1. Parameters
    # -----------------------
    params = sum(p.numel() for p in model.parameters()) / 1e6

    # -----------------------
    # 2. FLOPs
    # -----------------------
    try:
        flops, _ = profile(model, inputs=(dummy_input,), verbose=False)
        flops = flops / 1e9
    except:
        flops = 0.0  # fallback if unsupported

    # -----------------------
    # 3. Model Size
    # -----------------------
    torch.save(model.state_dict(), "temp.pth")
    size_mb = os.path.getsize("temp.pth") / (1024 * 1024)
    os.remove("temp.pth")

    # -----------------------
    # 4. Inference Time (FIXED)
    # -----------------------
    repetitions = 100
    warmup = 10

    with torch.no_grad():
        # Warm-up (VERY IMPORTANT)
        for _ in range(warmup):
            _ = model(dummy_input)

        if device.type == "cuda":
            starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            timings = []

            for _ in range(repetitions):
                starter.record()
                _ = model(dummy_input)
                ender.record()
                torch.cuda.synchronize()
                timings.append(starter.elapsed_time(ender))

            inf_time = sum(timings) / len(timings)

        else:
            start = time.time()
            for _ in range(repetitions):
                _ = model(dummy_input)
            end = time.time()

            inf_time = ((end - start) / repetitions) * 1000  # ms

    # -----------------------
    # 5. GPU Memory (FIXED)
    # -----------------------
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            _ = model(dummy_input)
        mem = torch.cuda.max_memory_allocated() / (1024 ** 3)
    else:
        mem = 0.0

    return {
        "Params(M)": round(params, 2),
        "FLOPs(G)": round(flops, 2),
        "Size(MB)": round(size_mb, 2),
        "Inf Time(ms)": round(inf_time, 2),
        "GPU Mem(GB)": round(mem, 2)
    }

In [ ]:
from torchvision.models import efficientnet_b4

model = efficientnet_b4(weights=None)
stats = compute_model_stats(model, (1,3,224,224))
print(stats)